# 10b - Silver: UN Comtrade — Normalize, Aggregate, and Coverage Audit

Reads `bronze.comtrade_hs6_raw` and produces the bilateral Comtrade tables
that merge with IMTS/DOTS in notebook 10.

## Outputs

| Table | Grain | Purpose |
|-------|-------|---------|
| `silver.comtrade_hs6_normalized` | reporter × partner × year × flow × hs6_code | Clean HS6 rows, deduplicated |
| `silver.comtrade_partner_annual` | reporter × partner × year × flow_type | Partner-annual totals (merges with DOTS in notebook 10) |
| `silver.comtrade_country_year_coverage` | country × year | Bilateral quality flags driving source-priority merge in notebook 10 |

Product-structure tables are intentionally owned by `scripts/load_comtrade_silver.py`,
which reads W00 national-total rows and writes `silver.comtrade_hs2_annual_w00`
and `silver.comtrade_product_coverage`.

## Pipeline position

```
03_bronze_comtrade_extract
        ↓
10b_silver_comtrade_normalize  ← this notebook
        ↓
10_silver_trade_partner_annual (reads comtrade_partner_annual + comtrade_country_year_coverage)
```

## Quality flag logic

| Flag | Meaning | Action in notebook 10 |
|------|---------|----------------------|
| `good` | ≥500 HS6 rows | Use Comtrade directly |
| `sparse` | 1–499 rows | Flag or fallback to IMTS |
| `partial_submission` | Known partial year (e.g. CAF 2006) | Exclude or fallback |
| `data_integrity_anomaly` | e.g. GHA 2004 (≤2 rows) | Exclude from default gold |
| `late_release` | 2024 rows not yet published | Treat as not-yet-available |
| `comtrade_dark` | Known zero-submission countries | Use IMTS/mirror only |
| `missing` | No rows | Fallback to IMTS |


In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

CATALOG = "cemac_ecowas_aes_trade"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

START_YEAR = 1990
END_YEAR = 2024
WORLD_PARTNER_CODE = "W00"
# Comtrade rows with fewer than this many HS6 lines per reporter×year are treated as sparse
SPARSE_THRESHOLD = 500
loaded_at = datetime.now(timezone.utc)

print("Outputs:")
print("  silver.comtrade_hs6_normalized      (reporter×partner×year×flow×hs6)")
print("  silver.comtrade_partner_annual       (reporter×partner×year×flow_type)")
print("  silver.comtrade_country_year_coverage (bilateral country×year)")
print("Product W00 HS2 tables are owned by scripts/load_comtrade_silver.py")


In [ ]:
# Cell 2 - Input checks
def table_exists(schema, table):
    return spark.sql(f"SHOW TABLES IN {schema} LIKE '{table}'").count() > 0


required = [("bronze", "comtrade_hs6_raw"), ("silver", "dim_country")]
missing = [f"{s}.{t}" for s, t in required if not table_exists(s, t)]
if missing:
    raise ValueError(f"Missing required tables: {missing}")

bronze_count = spark.table("bronze.comtrade_hs6_raw").count()
if bronze_count == 0:
    raise RuntimeError(
        "bronze.comtrade_hs6_raw is empty. Refusing to rebuild Comtrade silver tables, "
        "because this notebook drops and overwrites silver.comtrade_* outputs. "
        "Load verified Comtrade HS rows first or preserve the existing HS2 silver table."
    )
print(f"bronze.comtrade_hs6_raw rows: {bronze_count:,}")

print("Input coverage in bronze.comtrade_hs6_raw:")
spark.sql("""
    SELECT
        reporter_iso3,
        SUM(CASE WHEN NOT COALESCE(is_aggregate, false) THEN 1 ELSE 0 END) AS hs6_line_rows,
        SUM(CASE WHEN COALESCE(is_aggregate, false) THEN 1 ELSE 0 END) AS aggregate_rows,
        COUNT(DISTINCT CASE WHEN NOT COALESCE(is_aggregate, false) AND partner_iso3 != 'W00' THEN partner_iso3 END) AS bilateral_partners,
        COUNT(DISTINCT CASE WHEN NOT COALESCE(is_aggregate, false) THEN cmd_code END) AS hs6_codes,
        MIN(year) AS first_year,
        MAX(year) AS last_year
    FROM bronze.comtrade_hs6_raw
    GROUP BY reporter_iso3
    ORDER BY hs6_line_rows DESC
""").show(25, truncate=False)


In [ ]:
# Cell 3 - Normalize bronze HS6 rows → silver.comtrade_hs6_normalized
# Grain: reporter_iso3 + partner_iso3 + year + flow_type + hs6_code
# Silver filters: is_aggregate = false (subtotals kept in bronze are dropped here)
#                 partner_iso3 != W00 (world-total rows excluded from bilateral grain)

raw_bronze = spark.table("bronze.comtrade_hs6_raw")

silver_hs6 = (
    raw_bronze
    .filter(~F.coalesce(F.col("is_aggregate"), F.lit(False)))
    .filter(F.col("partner_iso3") != WORLD_PARTNER_CODE)
    .withColumn("hs6_code", F.lpad(F.col("cmd_code").cast("string"), 6, "0"))
    .withColumn("hs2_code", F.substring(F.col("hs6_code"), 1, 2))
    .withColumn(
        "flow_type",
        F.when(F.col("flow") == "X", F.lit("export"))
         .when(F.col("flow") == "M", F.lit("import"))
         .otherwise(F.lit("other")),
    )
    .withColumn("source_name", F.lit("UN_COMTRADE"))
)

# Deduplicate: sum values for any duplicate grain keys
silver_hs6_agg = (
    silver_hs6
    .groupBy(
        "reporter_iso3", "partner_iso3", "year", "flow_type",
        "hs6_code", "hs2_code", "source_name",
    )
    .agg(
        F.first(F.col("cmd_desc"), ignorenulls=True).alias("hs6_description"),
        F.lit(None).cast("string").alias("hs2_description"),   # enrichable later from HS lookup
        F.sum("value_usd").alias("trade_value_usd"),
        F.sum("net_wgt_kg").alias("net_weight_kg"),
        F.sum("qty").alias("quantity"),
        F.first("qty_unit", ignorenulls=True).alias("quantity_unit"),
        F.lit("raw").alias("quality_flag"),
        F.current_timestamp().alias("created_at"),
    )
)

row_count = silver_hs6_agg.count()
print(f"silver.comtrade_hs6_normalized rows: {row_count:,}")
silver_hs6_agg.groupBy("flow_type").count().show()

spark.sql("DROP TABLE IF EXISTS silver.comtrade_hs6_normalized")
(
    silver_hs6_agg.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("reporter_iso3", "year")
    .saveAsTable("silver.comtrade_hs6_normalized")
)
print("Write complete: silver.comtrade_hs6_normalized")


In [ ]:
# Cell 4 - Aggregate HS6 → silver.comtrade_partner_annual
# Grain: reporter_iso3 + partner_iso3 + year + flow_type
# This table is what notebook 10 compares/merges with IMTS/DOTS.

hs6 = spark.table("silver.comtrade_hs6_normalized")

comtrade_partner_annual = (
    hs6
    .groupBy("reporter_iso3", "partner_iso3", "year", "flow_type")
    .agg(
        F.sum("trade_value_usd").alias("trade_value_usd"),
        F.countDistinct("hs6_code").alias("distinct_hs6_products"),
        F.count("*").alias("comtrade_row_count"),
        F.lit("UN_COMTRADE_DIRECT").alias("source_method"),
        F.current_timestamp().alias("created_at"),
    )
)

print(f"silver.comtrade_partner_annual rows: {comtrade_partner_annual.count():,}")
comtrade_partner_annual.groupBy("flow_type").count().show()
print("Top 10 reporters by partner rows:")
(
    comtrade_partner_annual
    .groupBy("reporter_iso3")
    .agg(
        F.count("*").alias("partner_flow_rows"),
        F.countDistinct("partner_iso3").alias("distinct_partners"),
        F.sum("trade_value_usd").alias("total_trade_usd"),
    )
    .orderBy(F.desc("partner_flow_rows"))
).show(10, truncate=False)

spark.sql("DROP TABLE IF EXISTS silver.comtrade_partner_annual")
(
    comtrade_partner_annual.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("reporter_iso3", "year")
    .saveAsTable("silver.comtrade_partner_annual")
)
print("Write complete: silver.comtrade_partner_annual")


In [ ]:
# Cell 5 - Product HS2 ownership note
# This notebook owns bilateral Comtrade tables for the partner-dependency layer.
# Product-structure HS2 national totals are W00-only and are built by
# scripts/load_comtrade_silver.py into silver.comtrade_hs2_annual_w00 and
# silver.comtrade_product_coverage.

print("Skipping product HS2 writes in notebook 10b; W00 product tables are script-owned.")


In [ ]:
# Cell 6 - Country×year Comtrade coverage audit → silver.comtrade_country_year_coverage
# This table is read by notebook 10 to decide which reporter×years use Comtrade vs IMTS/DOTS.

hs6 = spark.table("silver.comtrade_hs6_normalized")

# Aggregate row counts per reporter×year (excluding W00 already filtered in Cell 3)
coverage_raw = (
    hs6
    .groupBy("reporter_iso3", "year")
    .agg(
        F.count("*").alias("comtrade_row_count"),
        F.countDistinct("partner_iso3").alias("distinct_partners"),
        F.countDistinct("hs6_code").alias("distinct_hs6_products"),
    )
    .withColumnRenamed("reporter_iso3", "country_iso3")
)

# Build the full country×year grid (21 countries × 35 years = 735 cells)
countries = spark.table("silver.dim_country").select("country_iso3")
years_df = (
    spark.range(START_YEAR, END_YEAR + 1)
    .withColumnRenamed("id", "year")
    .withColumn("year", F.col("year").cast("int"))
)
country_years = countries.crossJoin(years_df)

# The 9 confirmed Comtrade-dark countries (zero bilateral submission in all years)
COMTRADE_DARK = ["GNQ", "GNB", "LBR", "MLI", "NER", "NGA", "SEN", "SLE", "TGO"]

coverage_full = (
    country_years
    .join(coverage_raw, ["country_iso3", "year"], "left")
    .fillna({"comtrade_row_count": 0, "distinct_partners": 0, "distinct_hs6_products": 0})
    .withColumn("has_comtrade_data", F.col("comtrade_row_count") > 0)
    # Base status
    .withColumn(
        "comtrade_status",
        F.when(F.col("comtrade_row_count") == 0, F.lit("missing"))
         .when(F.col("comtrade_row_count") < SPARSE_THRESHOLD, F.lit("sparse"))
         .otherwise(F.lit("good")),
    )
    # Specific quality flags from gap analysis (ordered from most specific to least)
    .withColumn(
        "quality_flag",
        F.when(
            (F.col("country_iso3") == "GHA")
            & (F.col("year") == 2004)
            & (F.col("comtrade_row_count") <= 2),
            F.lit("data_integrity_anomaly"),
        )
        .when(
            (F.col("country_iso3") == "CAF")
            & (F.col("year") == 2006)
            & (F.col("comtrade_row_count") < SPARSE_THRESHOLD),
            F.lit("partial_submission"),
        )
        .when(
            (F.col("year") == END_YEAR) & (F.col("comtrade_row_count") == 0),
            F.lit("late_release"),
        )
        .when(
            F.col("country_iso3").isin(COMTRADE_DARK) & (F.col("comtrade_row_count") == 0),
            F.lit("comtrade_dark"),
        )
        .otherwise(F.col("comtrade_status")),
    )
    # Recommended action for notebook 10 and dashboard logic
    .withColumn(
        "recommended_action",
        F.when(F.col("quality_flag") == "good", F.lit("use_direct_comtrade"))
         .when(F.col("quality_flag") == "sparse", F.lit("flag_or_fallback"))
         .when(F.col("quality_flag") == "partial_submission", F.lit("exclude_or_fallback"))
         .when(F.col("quality_flag") == "data_integrity_anomaly", F.lit("exclude_from_default_gold"))
         .when(F.col("quality_flag") == "late_release", F.lit("treat_as_not_yet_available"))
         .when(F.col("quality_flag") == "comtrade_dark", F.lit("use_imts_mirror_only"))
         .otherwise(F.lit("use_imts_or_mirror")),
    )
    .withColumn("created_at", F.current_timestamp())
)

print("Coverage by quality_flag:")
coverage_full.groupBy("quality_flag").count().orderBy(F.desc("count")).show()

print(f"\nCountry breakdown for year {END_YEAR}:")
coverage_full.where(F.col("year") == END_YEAR).orderBy("country_iso3").show(25, truncate=False)

spark.sql("DROP TABLE IF EXISTS silver.comtrade_country_year_coverage")
(
    coverage_full.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.comtrade_country_year_coverage")
)
print("Write complete: silver.comtrade_country_year_coverage")


In [ ]:
# Cell 7 - Output summary and readiness check
print("=== 10b output tables ===")
for tbl in [
    "silver.comtrade_hs6_normalized",
    "silver.comtrade_partner_annual",
    "silver.comtrade_country_year_coverage",
]:
    cnt = spark.table(tbl).count()
    print(f"  {tbl}: {cnt:,} rows")

print("\nComtrade-ready countries (quality_flag = 'good', any year):")
spark.sql("""
    SELECT
        country_iso3,
        COUNT(*) AS good_years,
        MIN(year) AS first_good_year,
        MAX(year) AS last_good_year
    FROM silver.comtrade_country_year_coverage
    WHERE quality_flag = 'good'
    GROUP BY country_iso3
    ORDER BY good_years DESC
""").show(25, truncate=False)

print("Comtrade-dark countries (all years missing):")
spark.sql("""
    SELECT country_iso3, quality_flag, COUNT(*) AS dark_years
    FROM silver.comtrade_country_year_coverage
    WHERE quality_flag = 'comtrade_dark'
    GROUP BY country_iso3, quality_flag
    ORDER BY country_iso3
""").show(truncate=False)

print("Product HS2 national-total tables:")
print("  silver.comtrade_hs2_annual_w00 and silver.comtrade_product_coverage are owned by scripts/load_comtrade_silver.py")
